In [44]:
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
import requests
import math
import numpy as np

from pathlib import Path
from rasterio.merge import merge
from pyproj import Transformer
from shapely.geometry import box
from shapely.geometry import LineString, MultiLineString
from shapely.geometry import Point


In [2]:
# Assuming notebook working directory is notebooks/
PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RAW_DIR = PROJECT_DIR / "data" / "raw"

roads_path = PROCESSED_DIR / "roads_classified.gpkg"
admin_path = PROCESSED_DIR / "administrative_boundaries.gpkg"

assert roads_path.exists(), roads_path
assert admin_path.exists(), admin_path


In [8]:
roads = gpd.read_file(
    roads_path,
    layer="roads_classified"
)

districts = gpd.read_file(
    admin_path,
    layer="districts"
)

regions = gpd.read_file(
    admin_path,
    layer="regions"
)

unassigned = gpd.read_file(
    admin_path,
    layer="unassigned_areas"
)

print(f"Roads: {len(roads):,}")
print(f"Districts: {len(districts)}")
print(f"Regions: {len(regions)}")
print(f"Unassigned areas: {len(unassigned)}")

Roads: 571,899
Districts: 203
Regions: 14
Unassigned areas: 3


In [9]:
boolean_columns = [
    "is_confirmed_asphalt",
    "is_conventional",
    "is_potentially_drivable",
    "is_access_restricted",
    "is_4wd_only",
    "is_rough_track",
    "surface_known",
]

for col in boolean_columns:
    if col in roads.columns:
        roads[col] = (
            roads[col]
            .astype("string")
            .str.lower()
            .map({
                "true": True,
                "false": False,
                "1": True,
                "0": False,
            })
            .astype("boolean")
        )

In [10]:
layers = {
    "roads": roads,
    "districts": districts,
    "regions": regions,
}

for name, gdf in layers.items():
    print(f"\n{name.upper()}")
    print("CRS:", gdf.crs)
    print("Records:", len(gdf))
    print("Invalid:", (~gdf.is_valid).sum())
    print("Empty:", gdf.geometry.is_empty.sum())

# Verify metadata from our reconstructed boundaries
display(
    districts.loc[
        ~districts["is_complete"],
        ["name", "region_name", "boundary_source"]
    ]
)


ROADS
CRS: EPSG:4326
Records: 571899
Invalid: 0
Empty: 0

DISTRICTS
CRS: EPSG:4326
Records: 203
Invalid: 0
Empty: 0

REGIONS
CRS: EPSG:4326
Records: 14
Invalid: 0
Empty: 0


,name,region_name,boundary_source
202,Fargʻona tumani,Farg'ona Viloyati,partial_polygonize_from_relation_ways


In [18]:
# Exact OSM relation ID from our boundaries
bostanlyk = districts.loc[
    districts["osm_id"] == "2120216"
].copy()

assert len(bostanlyk) == 1

bost_geom = bostanlyk.geometry.iloc[0]

# Select road candidates using spatial index
indices = roads.sindex.query(
    bost_geom,
    predicate="intersects"
)

roads_bost = roads.iloc[indices].copy()

# Clip road segments to the district
roads_bost = gpd.clip(
    roads_bost,
    bostanlyk
)

print("Bostanlyk road segments:", len(roads_bost))
print(
    "Confirmed asphalt:",
    roads_bost["is_confirmed_asphalt"].sum()
)

print(
    roads_bost["highway"]
    .value_counts(dropna=False)
    .head(15)
    .to_string()
)

Bostanlyk road segments: 8840
Confirmed asphalt: 138
highway
living_street     2486
track             2446
service           1867
unclassified       875
residential        480
primary            186
secondary          148
trunk              144
tertiary           135
trunk_link          55
primary_link        16
secondary_link       2


In [19]:
print("Bostanlyk bounds (WGS84):")
print(bostanlyk.to_crs(4326).total_bounds)

Bostanlyk bounds (WGS84):
[69.5863291 41.2239141 71.2690927 42.2950775]


In [20]:

DEM_DIR = RAW_DIR / "dem"
DEM_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = (
    "https://copernicus-dem-30m"
    ".s3.eu-central-1.amazonaws.com"
)

west, south, east, north = bostanlyk.to_crs(4326).total_bounds

tiles = []

for lat in range(math.floor(south), math.ceil(north)):
    for lon in range(math.floor(west), math.ceil(east)):
        tile = f"Copernicus_DSM_COG_10_N{lat:02d}_00_E{lon:03d}_00_DEM"
        tiles.append(tile)

print("Tiles to download:", len(tiles))

for tile in tiles:
    filename = f"{tile}.tif"
    output = DEM_DIR / filename

    if output.exists():
        print("Already exists:", filename)
        continue

    url = f"{BASE_URL}/{tile}/{filename}"

    print("Downloading:", filename)

    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()

        with open(output, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

print("Download complete")

Tiles to download: 6
Downloading: Copernicus_DSM_COG_10_N41_00_E069_00_DEM.tif
Downloading: Copernicus_DSM_COG_10_N41_00_E070_00_DEM.tif
Downloading: Copernicus_DSM_COG_10_N41_00_E071_00_DEM.tif
Downloading: Copernicus_DSM_COG_10_N42_00_E069_00_DEM.tif
Downloading: Copernicus_DSM_COG_10_N42_00_E070_00_DEM.tif
Downloading: Copernicus_DSM_COG_10_N42_00_E071_00_DEM.tif
Download complete


In [21]:
dem_files = sorted(DEM_DIR.glob("*.tif"))

for path in dem_files:
    with rasterio.open(path) as src:
        print(
            path.name,
            "| CRS:", src.crs,
            "| Size:", src.width, "x", src.height,
            "| Resolution:", src.res,
            "| Bounds:", src.bounds
        )

Copernicus_DSM_COG_10_N41_00_E069_00_DEM.tif | CRS: EPSG:4326 | Size: 3600 x 3600 | Resolution: (0.0002777777777777778, 0.0002777777777777778) | Bounds: BoundingBox(left=68.99986111111112, bottom=41.00013888888889, right=69.99986111111112, top=42.00013888888889)
Copernicus_DSM_COG_10_N41_00_E070_00_DEM.tif | CRS: EPSG:4326 | Size: 3600 x 3600 | Resolution: (0.0002777777777777778, 0.0002777777777777778) | Bounds: BoundingBox(left=69.99986111111112, bottom=41.00013888888889, right=70.99986111111112, top=42.00013888888889)
Copernicus_DSM_COG_10_N41_00_E071_00_DEM.tif | CRS: EPSG:4326 | Size: 3600 x 3600 | Resolution: (0.0002777777777777778, 0.0002777777777777778) | Bounds: BoundingBox(left=70.99986111111112, bottom=41.00013888888889, right=71.99986111111112, top=42.00013888888889)
Copernicus_DSM_COG_10_N42_00_E069_00_DEM.tif | CRS: EPSG:4326 | Size: 3600 x 3600 | Resolution: (0.0002777777777777778, 0.0002777777777777778) | Bounds: BoundingBox(left=68.99986111111112, bottom=42.000138888888

In [24]:

DEM_OUTPUT = PROCESSED_DIR / "bostanlyk_dem.tif"

sources = [rasterio.open(path) for path in dem_files]

try:
    bounds = tuple(bostanlyk.to_crs(4326).total_bounds)

    mosaic, transform = merge(
        sources,
        bounds=bounds,
        nodata=-9999,
    )

    profile = sources[0].profile.copy()
    profile.update(
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        transform=transform,
        count=1,
        nodata=-9999,
        compress="deflate",
        tiled=True,
    )

    with rasterio.open(DEM_OUTPUT, "w", **profile) as dst:
        dst.write(mosaic)

finally:
    for src in sources:
        src.close()

print("DEM mosaic:", DEM_OUTPUT)

DEM mosaic: C:\Users\m.maxsudov\30daymapchallenge-2026\projects\day02_lines\data\processed\bostanlyk_dem.tif


In [27]:

SAMPLE_STEP_M = 30

roads_metric = roads_bost.to_crs("EPSG:32642")

transformer = Transformer.from_crs(
    "EPSG:32642", "EPSG:4326",
    always_xy=True
)

def iter_lines(geometry):
    if geometry.geom_type == "LineString":
        yield geometry
    elif geometry.geom_type == "MultiLineString":
        yield from geometry.geoms
    elif geometry.geom_type == "GeometryCollection":
        for part in geometry.geoms:
            yield from iter_lines(part)

samples = []

for idx, row in roads_metric.iterrows():
    for line in iter_lines(row.geometry):
        if line.is_empty:
            continue

        n_intervals = max(
            1, int(np.ceil(line.length / SAMPLE_STEP_M))
        )

        for distance in np.linspace(
            0, line.length, n_intervals + 1
        ):
            point = line.interpolate(distance)

            lon, lat = transformer.transform(
                point.x, point.y
            )

            samples.append({
                "road_index": idx,
                "lon": lon,
                "lat": lat,
                "highway": row["highway"],
                "is_asphalt": row["is_confirmed_asphalt"],
                "is_conventional": row["is_conventional"],
                "is_restricted": row["is_access_restricted"],
            })

print(f"Generated {len(samples):,} sampling points")

Generated 144,024 sampling points


In [28]:

sampling_df = pd.DataFrame(samples)

elevations = np.full(
    len(sampling_df), np.nan, dtype="float32"
)

BATCH_SIZE = 10_000

with rasterio.open(DEM_OUTPUT) as dem:
    for start in range(0, len(sampling_df), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(sampling_df))

        batch = sampling_df.iloc[start:end]

        coords = list(zip(
            batch["lon"],
            batch["lat"]
        ))

        values = np.array(
            [v[0] for v in dem.sample(coords, masked=True)],
            dtype="float32"
        )

        elevations[start:end] = values

    nodata = dem.nodata

if nodata is not None:
    elevations[elevations == nodata] = np.nan

elevations[~np.isfinite(elevations)] = np.nan

sampling_df["elevation_m"] = elevations

print("Valid elevation points:", sampling_df["elevation_m"].notna().sum())
print("Missing elevation points:", sampling_df["elevation_m"].isna().sum())
print("Elevation range:", sampling_df["elevation_m"].min(),
      sampling_df["elevation_m"].max())

Valid elevation points: 144024
Missing elevation points: 0
Elevation range: 566.8098 2664.698


In [29]:
asphalt = sampling_df[
    sampling_df["is_asphalt"].eq(True)
]

conventional = sampling_df[
    sampling_df["is_conventional"].eq(True)
]

all_drivable = sampling_df

categories = {
    "Confirmed asphalt": asphalt,
    "Conventional roads": conventional,
    "All potentially drivable": all_drivable,
}

results = []

for category, df in categories.items():
    valid = df.dropna(subset=["elevation_m"])

    if valid.empty:
        continue

    highest = valid.loc[valid["elevation_m"].idxmax()]

    results.append({
        "category": category,
        "max_elevation_m": round(highest["elevation_m"], 1),
        "highway": highest["highway"],
        "longitude": highest["lon"],
        "latitude": highest["lat"],
        "road_index": highest["road_index"],
        "access_restricted": highest["is_restricted"],
    })

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

                category  max_elevation_m      highway  longitude  latitude  road_index  access_restricted
       Confirmed asphalt      1736.599976      primary  69.996857 41.518410       24947              False
      Conventional roads      1863.900024 unclassified  70.290812 41.478149       59915              False
All potentially drivable      2664.699951        track  70.040267 41.502422      540882              False


In [33]:

districts_wgs = districts.to_crs(4326)

west, south, east, north = districts_wgs.total_bounds

tiles_info = []

for lat in range(math.floor(south), math.ceil(north)):
    for lon in range(math.floor(west), math.ceil(east)):

        tile_geom = box(lon, lat, lon + 1, lat + 1)

        # Include only tiles intersecting administrative districts
        if not districts_wgs.intersects(tile_geom).any():
            continue

        tile_name = (
            f"Copernicus_DSM_COG_10_"
            f"N{lat:02d}_00_E{lon:03d}_00_DEM"
        )

        path = DEM_DIR / f"{tile_name}.tif"

        tiles_info.append({
            "tile": tile_name,
            "lat": lat,
            "lon": lon,
            "downloaded": path.exists(),
            "path": str(path),
        })

tile_df = pd.DataFrame(tiles_info)

print("Required tiles:", len(tile_df))
print("Already downloaded:", tile_df["downloaded"].sum())
print("Remaining:", (~tile_df["downloaded"]).sum())

print(tile_df.to_string(index=False))

Required tiles: 83
Already downloaded: 5
Remaining: 78
                                    tile  lat  lon  downloaded                                                                                                                      path
Copernicus_DSM_COG_10_N37_00_E066_00_DEM   37   66       False C:\Users\m.maxsudov\30daymapchallenge-2026\projects\day02_lines\data\raw\dem\Copernicus_DSM_COG_10_N37_00_E066_00_DEM.tif
Copernicus_DSM_COG_10_N37_00_E067_00_DEM   37   67       False C:\Users\m.maxsudov\30daymapchallenge-2026\projects\day02_lines\data\raw\dem\Copernicus_DSM_COG_10_N37_00_E067_00_DEM.tif
Copernicus_DSM_COG_10_N37_00_E068_00_DEM   37   68       False C:\Users\m.maxsudov\30daymapchallenge-2026\projects\day02_lines\data\raw\dem\Copernicus_DSM_COG_10_N37_00_E068_00_DEM.tif
Copernicus_DSM_COG_10_N38_00_E064_00_DEM   38   64       False C:\Users\m.maxsudov\30daymapchallenge-2026\projects\day02_lines\data\raw\dem\Copernicus_DSM_COG_10_N38_00_E064_00_DEM.tif
Copernicus_DSM_COG_1

In [31]:
BASE_URL = (
    "https://copernicus-dem-30m"
    ".s3.eu-central-1.amazonaws.com"
)

def download_dem_tiles(tile_df, limit=10):
    pending = tile_df.loc[~tile_df["downloaded"]].head(limit)

    for _, row in pending.iterrows():
        tile = row["tile"]
        output = Path(row["path"])
        url = f"{BASE_URL}/{tile}/{tile}.tif"
        temp = output.with_suffix(".tif.part")

        print("Downloading:", tile)

        try:
            with requests.get(
                url, stream=True, timeout=(20, 180)
            ) as response:
                response.raise_for_status()

                with open(temp, "wb") as f:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)

            # Validate before finalizing
            with rasterio.open(temp) as src:
                assert src.count == 1
                assert src.crs is not None

            temp.replace(output)
            print("Saved:", output.name)

        except Exception as e:
            temp.unlink(missing_ok=True)
            print("Failed:", tile, str(e))

    tile_df["downloaded"] = tile_df["path"].map(
        lambda p: Path(p).exists()
    )

    print(
        "Progress:",
        int(tile_df["downloaded"].sum()),
        "/",
        len(tile_df)
    )

In [39]:
download_dem_tiles(tile_df, limit=15)

Downloading: Copernicus_DSM_COG_10_N43_00_E065_00_DEM
Saved: Copernicus_DSM_COG_10_N43_00_E065_00_DEM.tif
Downloading: Copernicus_DSM_COG_10_N44_00_E055_00_DEM
Saved: Copernicus_DSM_COG_10_N44_00_E055_00_DEM.tif
Downloading: Copernicus_DSM_COG_10_N44_00_E056_00_DEM
Saved: Copernicus_DSM_COG_10_N44_00_E056_00_DEM.tif
Downloading: Copernicus_DSM_COG_10_N44_00_E057_00_DEM
Saved: Copernicus_DSM_COG_10_N44_00_E057_00_DEM.tif
Downloading: Copernicus_DSM_COG_10_N44_00_E058_00_DEM
Saved: Copernicus_DSM_COG_10_N44_00_E058_00_DEM.tif
Downloading: Copernicus_DSM_COG_10_N44_00_E059_00_DEM
Saved: Copernicus_DSM_COG_10_N44_00_E059_00_DEM.tif
Downloading: Copernicus_DSM_COG_10_N44_00_E060_00_DEM
Saved: Copernicus_DSM_COG_10_N44_00_E060_00_DEM.tif
Downloading: Copernicus_DSM_COG_10_N44_00_E061_00_DEM
Saved: Copernicus_DSM_COG_10_N44_00_E061_00_DEM.tif
Downloading: Copernicus_DSM_COG_10_N45_00_E055_00_DEM
Saved: Copernicus_DSM_COG_10_N45_00_E055_00_DEM.tif
Downloading: Copernicus_DSM_COG_10_N45_00_E056

In [40]:
from pathlib import Path
import rasterio

validation = []

for _, row in tile_df.iterrows():
    path = Path(row["path"])

    status = "missing"

    if path.exists():
        try:
            with rasterio.open(path) as src:
                status = (
                    "valid"
                    if src.count == 1
                    and src.crs is not None
                    and src.width > 0
                    and src.height > 0
                    else "invalid"
                )
        except Exception:
            status = "invalid"

    validation.append({
        "tile": row["tile"],
        "status": status,
        "size_mb": round(path.stat().st_size / 1e6, 1)
                   if path.exists() else None,
    })

validation_df = pd.DataFrame(validation)

print(validation_df["status"].value_counts())
print("Total size (GB):", round(
    validation_df["size_mb"].sum() / 1000, 2
))

status
valid    83
Name: count, dtype: int64
Total size (GB): 3.44


In [41]:
test_tile = DEM_DIR / (
    "Copernicus_DSM_COG_10_N41_00_E070_00_DEM.tif"
)

with rasterio.open(test_tile) as src:
    tile_bounds = src.bounds

tile_polygon = box(*tile_bounds)

# Roads must be in EPSG:4326
roads_wgs = roads.to_crs(4326)

indices = roads_wgs.sindex.query(
    tile_polygon,
    predicate="intersects"
)

tile_roads = roads_wgs.iloc[indices].copy()

# Clip to the actual raster extent
tile_roads.geometry = tile_roads.geometry.intersection(
    tile_polygon
)

tile_roads = tile_roads.loc[
    ~tile_roads.geometry.is_empty
].copy()

print("Road segments in tile:", len(tile_roads))

Road segments in tile: 6423


In [43]:

STEP_M = 30
metric = tile_roads.to_crs("EPSG:32642")

transformer = Transformer.from_crs(
    "EPSG:32642", "EPSG:4326",
    always_xy=True
)

def iter_lines(geom):
    if geom.geom_type == "LineString":
        yield geom
    elif geom.geom_type == "MultiLineString":
        yield from geom.geoms
    elif geom.geom_type == "GeometryCollection":
        for part in geom.geoms:
            yield from iter_lines(part)

records = []

for idx, row in metric.iterrows():
    for line in iter_lines(row.geometry):
        if line.is_empty:
            continue

        n = max(1, int(np.ceil(line.length / STEP_M)))

        for distance in np.linspace(0, line.length, n + 1):
            p = line.interpolate(distance)
            lon, lat = transformer.transform(p.x, p.y)

            records.append({
                "osm_id": row["id"],
                "highway": row["highway"],
                "is_asphalt": str(
                    row["is_confirmed_asphalt"]
                ).lower() == "true",
                "is_conventional": str(
                    row["is_conventional"]
                ).lower() == "true",
                "is_restricted": str(
                    row["is_access_restricted"]
                ).lower() == "true",
                "lon": lon,
                "lat": lat,
            })

points = pd.DataFrame.from_records(records)

print("Sample points:", len(points))

Sample points: 118751


In [45]:
with rasterio.open(test_tile) as dem:
    elevations = np.array(
        [
            float(value[0]) if not np.ma.is_masked(value[0])
            else np.nan
            for value in dem.sample(
                zip(points["lon"], points["lat"]),
                masked=True
            )
        ],
        dtype="float32"
    )

points["elevation_m"] = elevations
points = points.dropna(subset=["elevation_m"]).copy()

points_gdf = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points.lon, points.lat),
    crs="EPSG:4326"
)

district_lookup = districts.to_crs(4326)[
    ["osm_id", "name", "geometry"]
].copy()

district_lookup = district_lookup.rename(columns={
    "osm_id": "district_id",
    "name": "district_name"
})

points_gdf = gpd.sjoin(
    points_gdf,
    district_lookup,
    how="left",
    predicate="intersects"
)

print("Valid samples:", len(points_gdf))
print(
    "Without district:",
    points_gdf["district_id"].isna().sum()
)

Valid samples: 118635
Without district: 2801


In [46]:
categories = {
    "asphalt": points_gdf["is_asphalt"],
    "conventional": points_gdf["is_conventional"],
    "all_drivable": pd.Series(
        True, index=points_gdf.index
    ),
}

results = []

for category, mask in categories.items():
    subset = points_gdf.loc[mask].copy()

    if subset.empty:
        continue

    # Keep only the highest sample per road and district
    candidates = (
        subset
        .sort_values("elevation_m", ascending=False)
        .drop_duplicates(["district_id", "osm_id"])
    )

    # Top 10 different roads per district
    top = candidates.groupby(
        "district_id", dropna=False
    ).head(10).copy()

    top["category"] = category
    results.append(top)

top_candidates = pd.concat(results, ignore_index=True)

summary = (
    top_candidates
    .groupby(
        ["district_id", "district_name", "category"],
        dropna=False
    )["elevation_m"]
    .max()
    .reset_index(name="max_elevation_m")
)

print(
    summary.sort_values(
        "max_elevation_m", ascending=False
    ).head(20).to_string(index=False)
)

district_id     district_name     category  max_elevation_m
   15551713        Pop tumani all_drivable      3420.014648
        NaN               NaN all_drivable      3265.651855
   15551713        Pop tumani conventional      3080.529541
    2120216 Bo'stonliq Tumani all_drivable      2656.953857
    5745852  Ohangaron Tumani all_drivable      2610.597656
    5745852  Ohangaron Tumani conventional      2610.597656
        NaN               NaN conventional      2335.797852
   15551713        Pop tumani      asphalt      2330.519775
    5745852  Ohangaron Tumani      asphalt      1960.058960
    5745823    Parkent Tumani all_drivable      1950.280029
    2120216 Bo'stonliq Tumani conventional      1857.110596
    2120216 Bo'stonliq Tumani      asphalt      1716.649292
   15551712      Chust tumani all_drivable      1573.491455
    7774377     Angren shahri all_drivable      1550.841309
    7774377     Angren shahri conventional      1499.360962
    7774377     Angren shahri      aspha

In [47]:
unassigned_points = points_gdf.loc[
    points_gdf["district_id"].isna()
]

print("Unassigned points:", len(unassigned_points))

print(
    unassigned_points[
        ["osm_id", "highway", "elevation_m", "lon", "lat"]
    ]
    .sort_values("elevation_m", ascending=False)
    .head(20)
    .to_string(index=False)
)

Unassigned points: 2801
   osm_id highway  elevation_m       lon       lat
266410404   track  3265.651855 70.784057 41.202163
266410404   track  3265.493408 70.785572 41.198894
266410404   track  3265.145020 70.785516 41.199161
266410404   track  3265.023926 70.783783 41.202662
266410404   track  3264.995605 70.785629 41.198627
266410404   track  3264.802734 70.784921 41.200981
266410404   track  3264.679932 70.783920 41.202412
266410404   track  3264.523682 70.785685 41.198361
266410404   track  3264.425537 70.784392 41.201686
266410404   track  3264.326904 70.784216 41.201921
266410404   track  3264.300293 70.785454 41.199427
266410404   track  3264.267822 70.785716 41.198094
266410404   track  3263.917236 70.784745 41.201216
266410404   track  3263.862549 70.783508 41.203161
266410404   track  3263.775879 70.785098 41.200746
266410404   track  3263.551270 70.783234 41.203659
266410404   track  3263.536865 70.784569 41.201451
266410404   track  3263.254395 70.783645 41.202911
2664104

In [48]:
# country уже загружен из административного GeoPackage
# Если нет, используй:
country = gpd.read_file(
    admin_path,
    layer="country"
).to_crs(4326)

unassigned = points_gdf.loc[
    points_gdf["district_id"].isna()
].copy()

country_geom = country.geometry.union_all()

unassigned["inside_uzbekistan"] = (
    unassigned.geometry.intersects(country_geom)
)

print("Unassigned samples:", len(unassigned))

print(
    unassigned["inside_uzbekistan"]
    .value_counts()
    .to_string()
)

print("\nHighest points inside Uzbekistan:")
print(
    unassigned.loc[unassigned["inside_uzbekistan"]]
    .nlargest(10, "elevation_m")[
        ["osm_id", "highway", "elevation_m", "lon", "lat"]
    ]
    .to_string(index=False)
)

print("\nHighest points outside Uzbekistan:")
print(
    unassigned.loc[~unassigned["inside_uzbekistan"]]
    .nlargest(10, "elevation_m")[
        ["osm_id", "highway", "elevation_m", "lon", "lat"]
    ]
    .to_string(index=False)
)

Unassigned samples: 2801
inside_uzbekistan
False    2801

Highest points inside Uzbekistan:
Empty DataFrame
Columns: [osm_id, highway, elevation_m, lon, lat]
Index: []

Highest points outside Uzbekistan:
   osm_id highway  elevation_m       lon       lat
266410404   track  3265.651855 70.784057 41.202163
266410404   track  3265.493408 70.785572 41.198894
266410404   track  3265.145020 70.785516 41.199161
266410404   track  3265.023926 70.783783 41.202662
266410404   track  3264.995605 70.785629 41.198627
266410404   track  3264.802734 70.784921 41.200981
266410404   track  3264.679932 70.783920 41.202412
266410404   track  3264.523682 70.785685 41.198361
266410404   track  3264.425537 70.784392 41.201686
266410404   track  3264.326904 70.784216 41.201921


In [49]:
points_gdf["territory_status"] = "assigned"

missing = points_gdf["district_id"].isna()

country_geom = country.geometry.union_all()

inside = points_gdf.geometry.intersects(country_geom)

points_gdf.loc[
    missing & inside,
    "territory_status"
] = "unassigned_inside"

points_gdf.loc[
    missing & ~inside,
    "territory_status"
] = "outside_or_unresolved"

print(
    points_gdf["territory_status"]
    .value_counts()
    .to_string()
)

territory_status
assigned                 115834
outside_or_unresolved      2801


In [55]:
import subprocess
import sys

script_path = PROJECT_DIR / "scripts" / "elevation_pipeline.py"

subprocess.run(
    [sys.executable, str(script_path)],
    check=True
)

CompletedProcess(args=['C:\\Users\\m.maxsudov\\30daymapchallenge-2026\\.venv\\Scripts\\python.exe', 'C:\\Users\\m.maxsudov\\30daymapchallenge-2026\\projects\\day02_lines\\scripts\\elevation_pipeline.py'], returncode=0)

In [56]:
results_path = (
    PROCESSED_DIR
    / "elevation"
    / "district_elevation_stats.csv"
)

national = pd.read_csv(results_path)

for category in [
    "asphalt",
    "conventional",
    "all_drivable"
]:
    print(f"\nTOP 10: {category.upper()}")

    top = national.loc[
        national["category"] == category
    ].nlargest(10, "max_elevation_m")

    print(
        top[
            [
                "district_name",
                "max_elevation_m",
                "highway",
                "osm_id"
            ]
        ].to_string(index=False)
    )


TOP 10: ASPHALT
    district_name  max_elevation_m      highway       osm_id
     Zomin Tumani        2446.5762      primary  529045942.0
       Pop tumani        2151.1868        trunk  244248110.0
 Ohangaron Tumani        1961.1823        trunk  244251953.0
Bo'stonliq Tumani        1729.6481      primary  172485328.0
     Urgut tumani        1666.6313        trunk   50372950.0
     Kitob tumani        1665.9722        trunk   50372950.0
 Sariosiyo Tumani        1584.7819 unclassified  419847473.0
Dehqonobod tumani        1581.9846        trunk  820391195.0
   Parkent Tumani        1566.4409     tertiary  493613451.0
   Qamashi tumani        1547.2800    secondary 1300222085.0

TOP 10: CONVENTIONAL
    district_name  max_elevation_m      highway       osm_id
       Pop tumani        3080.2732 unclassified  512550512.0
 Ohangaron Tumani        2902.1130 unclassified  247173701.0
   Qamashi tumani        2737.6885      service  557210851.0
      Uzun tumani        2663.5776     tertiar

In [57]:
print("Districts:", national["district_id"].nunique())

print(
    national.groupby("category")
    ["max_elevation_m"]
    .agg(["count", "min", "median", "max"])
    .to_string()
)

print(
    "\nDistrict-category combinations without measurements:",
    national["max_elevation_m"].isna().sum()
)

Districts: 203
              count       min     median        max
category                                           
all_drivable    203  71.23603  491.47754  3420.1345
asphalt         192  67.12718  442.23485  2446.5762
conventional    203  71.23603  488.18723  3080.2732

District-category combinations without measurements: 11


In [58]:
ELEVATION_DIR = PROCESSED_DIR / "elevation"

maxima = pd.read_csv(
    ELEVATION_DIR / "highest_road_points.csv",
    dtype={
        "osm_id": "string",
        "district_id": "string",
    }
)

country = gpd.read_file(
    admin_path,
    layer="country"
).to_crs(4326)

maxima_gdf = gpd.GeoDataFrame(
    maxima,
    geometry=gpd.points_from_xy(
        maxima["lon"], maxima["lat"]
    ),
    crs="EPSG:4326"
)

country_geom = country.geometry.union_all()

maxima_gdf["inside_country"] = (
    maxima_gdf.geometry.intersects(country_geom)
)

print("Total maxima:", len(maxima_gdf))

print(
    maxima_gdf["inside_country"]
    .value_counts()
    .to_string()
)

print("\nMaxima outside reconstructed country boundary:")

outside = maxima_gdf.loc[
    ~maxima_gdf["inside_country"]
].sort_values("elevation_m", ascending=False)

print(
    outside[
        [
            "district_name", "category",
            "elevation_m", "osm_id",
            "lon", "lat"
        ]
    ].head(30).to_string(index=False)
)

Total maxima: 598
inside_country
True    598

Maxima outside reconstructed country boundary:
Empty DataFrame
Columns: [district_name, category, elevation_m, osm_id, lon, lat]
Index: []


In [59]:
review = (
    maxima_gdf
    .sort_values("elevation_m", ascending=False)
    .groupby("category", group_keys=False)
    .head(10)
    .copy()
)

review["osm_url"] = (
    "https://www.openstreetmap.org/way/"
    + review["osm_id"].astype(str)
)

review = review[
    [
        "category",
        "district_name",
        "elevation_m",
        "highway",
        "osm_id",
        "is_restricted",
        "inside_country",
        "lon",
        "lat",
        "osm_url",
    ]
]

print(review.to_string(index=False))

review.to_csv(
    ELEVATION_DIR / "qa_review_top30.csv",
    index=False
)

    category     district_name  elevation_m      highway     osm_id  is_restricted  inside_country       lon       lat                                      osm_url
all_drivable        Pop tumani    3420.1345        track  266410404          False            True 70.805947 41.169025  https://www.openstreetmap.org/way/266410404
all_drivable      Zomin Tumani    3234.3152        track  862527332          False            True 68.535304 39.580433  https://www.openstreetmap.org/way/862527332
all_drivable Shahrisabz tumani    3120.6575        track  547984980          False            True 67.669653 39.124934  https://www.openstreetmap.org/way/547984980
conventional        Pop tumani    3080.2732 unclassified  512550512          False            True 70.682724 41.204978  https://www.openstreetmap.org/way/512550512
all_drivable  Ohangaron Tumani    2902.1130 unclassified  247173701           True            True 70.378456 40.919511  https://www.openstreetmap.org/way/247173701
conventional  Oh